# P00.차량전체 포함 데이터만 남기기 (이미지 ↔ 라벨 필터링)

- 데이터: AI-Hub "차량 외관 영상 데이터" (원천데이터=이미지, 라벨링데이터=json)
- 매칭 기준: **파일명(확장자 제외)이 동일** → 이미지 1개 ↔ json 1개
- 남길 기준: 라벨 json의 `objects` 안에 `"classId": "P00.차량전체"` 가 **하나라도 있으면 유지**, 없으면 제거

> ⚠️ 이 노트북은 파일을 **삭제/이동**합니다. 반드시 `DRY_RUN = True`(기본값)로 먼저 **미리보기**를 돌려
> 남길/지울 개수를 확인한 뒤, 이상 없을 때만 `DRY_RUN = False`로 바꿔 실제 실행하세요.
> 기본 모드는 `"move"`(지울 파일을 격리 폴더로 이동 → 언제든 복구 가능)입니다.

## 0. 환경 / 라이브러리

In [1]:
import os, sys, json, shutil
from pathlib import Path

from tqdm import tqdm

print("Python:", sys.version.split()[0])

Python: 3.11.9


## 1. 경로 & 옵션 설정 (여기만 수정)

- `IMAGE_ROOT` : 이미지 최상위 폴더 (예: ...\\1.Training\\원천데이터)
- `LABEL_ROOT` : 라벨(json) 최상위 폴더 (예: ...\\1.Training\\라벨링데이터)
- 두 폴더 아래로 TS1/TL1, 브랜드, 모델, 트림 등 몇 단계가 있어도 재귀적으로 전부 탐색합니다.

In [2]:
# ==== 여기 경로 두 줄만 실제 값으로 수정하세요 ====
IMAGE_ROOT = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터")
LABEL_ROOT = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\라벨링데이터")
# =================================================

# --- 필터 조건 ---
TARGET_CLASS = "P00.차량전체"     # 이 classId가 있는 json만 남김

# --- 안전 옵션 ---
DRY_RUN = True                    # True: 미리보기만 (파일 변경 없음) / False: 실제 실행
MODE = "move"                     # "move": 지울 파일을 격리 폴더로 이동(복구 가능) / "delete": 영구 삭제
REMOVE_ORPHAN_IMAGES = True       # 짝(json)이 아예 없는 이미지도 제거 대상에 포함할지

# --- 격리 폴더 (MODE="move"일 때만 사용) ---
QUARANTINE_ROOT = Path(r"C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터")

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

## 2. 파일 수집

In [3]:
def find_all_files(root: Path, exts: set):
    """root 아래 모든 하위 폴더를 재귀적으로 탐색해 지정 확장자 파일을 전부 반환."""
    if not root.exists():
        raise FileNotFoundError(f"경로가 존재하지 않습니다: {root}")
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]

image_files = find_all_files(IMAGE_ROOT, IMAGE_EXTS)
label_files = find_all_files(LABEL_ROOT, {".json"})

print(f"이미지 파일: {len(image_files):,}개")
print(f"라벨 파일:   {len(label_files):,}개")
print("이미지 예시:", image_files[0] if image_files else "없음")
print("라벨 예시:  ", label_files[0] if label_files else "없음")

이미지 파일: 257,740개
라벨 파일:   257,740개
이미지 예시: C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_002.jpg
라벨 예시:   C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\라벨링데이터\TL1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_002.json


## 3. 이미지 ↔ 라벨 매칭 (파일명 기준)

폴더 구조가 서로 달라도 **파일명(stem)이 같으면** 한 쌍으로 봅니다.
(예: `..._041.jpg` ↔ `..._041.json`)

In [4]:
label_lookup = {p.stem: p for p in label_files}

paired = []          # (이미지, 라벨) 짝이 맞은 것
orphan_images = []   # 짝(json)이 없는 이미지

for img in image_files:
    lbl = label_lookup.get(img.stem)
    if lbl is not None:
        paired.append((img, lbl))
    else:
        orphan_images.append(img)

paired_stems = {img.stem for img, _ in paired}
orphan_labels = [lbl for stem, lbl in label_lookup.items() if stem not in paired_stems]  # 이미지 없는 json

print(f"매칭된 쌍:        {len(paired):,}개")
print(f"짝 없는 이미지:   {len(orphan_images):,}개")
print(f"짝 없는 라벨:     {len(orphan_labels):,}개")

매칭된 쌍:        257,740개
짝 없는 이미지:   0개
짝 없는 라벨:     0개


## 4. 라벨 JSON에서 P00.차량전체 포함 여부 판정

AI-Hub json마다 구조가 조금씩 다를 수 있어, json 전체를 재귀 탐색해서 모든 `classId` 값을 모읍니다.
그 안에 `TARGET_CLASS`가 있으면 **유지 대상**입니다. (인코딩은 utf-8-sig/utf-8/cp949 순으로 자동 시도)

In [5]:
def load_json(path: Path):
    """인코딩을 자동 판별해 json을 읽는다."""
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return json.loads(path.read_text(encoding=enc))
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {path}")

def collect_class_ids(obj) -> set:
    """중첩된 dict/list를 재귀 탐색해 모든 classId 문자열을 수집."""
    found = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "classId" and isinstance(v, str):
                found.add(v.strip())
            else:
                found |= collect_class_ids(v)
    elif isinstance(obj, list):
        for item in obj:
            found |= collect_class_ids(item)
    return found

keep_pairs, remove_pairs, error_pairs = [], [], []

for img, lbl in tqdm(paired, desc="라벨 검사"):
    try:
        class_ids = collect_class_ids(load_json(lbl))
    except Exception as e:
        error_pairs.append((img, lbl, str(e)))
        continue
    if TARGET_CLASS in class_ids:
        keep_pairs.append((img, lbl))
    else:
        remove_pairs.append((img, lbl))

print(f"유지 (P00 있음): {len(keep_pairs):,}쌍")
print(f"제거 (P00 없음): {len(remove_pairs):,}쌍")
print(f"읽기 오류:       {len(error_pairs):,}쌍")
if error_pairs:
    print("  └ 오류 예시:", error_pairs[0][1], "→", error_pairs[0][2])

라벨 검사: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 257740/257740 [01:06<00:00, 3859.74it/s]

유지 (P00 있음): 89,506쌍
제거 (P00 없음): 168,234쌍
읽기 오류:       0쌍


## 5. 남길 파일 / 지울 파일 목록 확정 + 요약

- **유지**: P00 있는 이미지 + 그 짝 json
- **제거**: P00 없는 이미지 + 그 짝 json (+ 옵션에 따라 짝 없는 이미지)
- 읽기 오류가 난 쌍은 안전하게 **유지**로 둡니다.

In [6]:
# 제거 대상 모으기
remove_images = [img for img, _ in remove_pairs]
remove_labels = [lbl for _, lbl in remove_pairs]

if REMOVE_ORPHAN_IMAGES:
    remove_images += orphan_images

remove_files = remove_images + remove_labels
keep_files = [f for pair in keep_pairs for f in pair]  # 참고용 (건드리지 않음)

print("=" * 46)
print(f"{'구분':<20}{'개수':>12}")
print("-" * 46)
print(f"{'유지 이미지':<20}{len(keep_pairs):>12,}")
print(f"{'유지 라벨':<20}{len(keep_pairs):>12,}")
print(f"{'제거 이미지(P00없음)':<20}{len(remove_pairs):>12,}")
if REMOVE_ORPHAN_IMAGES:
    print(f"{'제거 이미지(짝없음)':<20}{len(orphan_images):>12,}")
print(f"{'제거 라벨(P00없음)':<20}{len(remove_labels):>12,}")
print("-" * 46)
print(f"{'>>> 총 제거 파일':<20}{len(remove_files):>12,}")
print("=" * 46)

print("\n제거 대상 예시 (최대 5개):")
for f in remove_files[:5]:
    print("  -", f)

구분                            개수
----------------------------------------------
유지 이미지                    89,506
유지 라벨                     89,506
제거 이미지(P00없음)            168,234
제거 이미지(짝없음)                    0
제거 라벨(P00없음)             168,234
----------------------------------------------
>>> 총 제거 파일              336,468

제거 대상 예시 (최대 5개):
  - C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_002.jpg
  - C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_003.jpg
  - C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_004.jpg
  - C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_005.jpg
  - C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_

## 6. 실제 정리 실행 (이동 / 삭제)

**`DRY_RUN = True`이면 아무 파일도 건드리지 않고** 무엇을 할지만 출력합니다.
실제로 정리하려면 위 셀 결과를 확인한 뒤 **1번 셀에서 `DRY_RUN = False`로 바꾸고** 이 셀을 다시 실행하세요.

- `MODE = "move"` : 제거 대상을 `QUARANTINE_ROOT` 아래로 **이동** (원본 폴더 구조 유지 → 나중에 되돌리기 쉬움)
- `MODE = "delete"` : 제거 대상을 **영구 삭제** (복구 불가)

In [7]:
def quarantine_dest(src: Path) -> Path:
    """원본이 IMAGE_ROOT/LABEL_ROOT 중 어디 소속인지 보고 상대경로를 유지해 격리 경로 생성."""
    for base, sub in ((IMAGE_ROOT, "원천데이터"), (LABEL_ROOT, "라벨링데이터")):
        try:
            rel = src.relative_to(base)
            return QUARANTINE_ROOT / sub / rel
        except ValueError:
            continue
    return QUARANTINE_ROOT / "기타" / src.name

if DRY_RUN:
    print(f"[미리보기] MODE={MODE} · 제거 대상 {len(remove_files):,}개")
    print("실제 실행하려면 1번 셀에서 DRY_RUN = False 로 바꾸세요.")
    for f in remove_files[:5]:
        if MODE == "move":
            print(f"  이동  {f}\n     → {quarantine_dest(f)}")
        else:
            print(f"  삭제  {f}")
else:
    ok, fail = 0, 0
    errors = []
    for f in tqdm(remove_files, desc=f"{MODE} 진행"):
        try:
            if not f.exists():
                continue
            if MODE == "move":
                dest = quarantine_dest(f)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(f), str(dest))
            elif MODE == "delete":
                f.unlink()
            else:
                raise ValueError(f"알 수 없는 MODE: {MODE}")
            ok += 1
        except Exception as e:
            fail += 1
            errors.append((f, str(e)))

    print(f"\n완료 · 성공 {ok:,}개 / 실패 {fail:,}개")
    if MODE == "move":
        print("격리 위치:", QUARANTINE_ROOT)
    if errors:
        print("실패 예시:", errors[0][0], "→", errors[0][1])

[미리보기] MODE=move · 제거 대상 336,468개
실제 실행하려면 1번 셀에서 DRY_RUN = False 로 바꾸세요.
  이동  C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_002.jpg
     → C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_002.jpg
  이동  C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_003.jpg
     → C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_003.jpg
  이동  C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_004.jpg
     → C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\사용데이터\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_211222_AU_006_17_BK_A_P_01_004.jpg
  이동  C:\Users\Win11Pro\Downloads\091.차량 외관 영상 데이터\01.데이터\1.Training\원천데이터\TS1\AU_아우디\006_A4\2017_검정_트림A\C_

## 7. (선택) 결과 재확인

In [8]:
remain_img = find_all_files(IMAGE_ROOT, IMAGE_EXTS)
remain_lbl = find_all_files(LABEL_ROOT, {".json"})
print(f"정리 후 남은 이미지: {len(remain_img):,}개")
print(f"정리 후 남은 라벨:   {len(remain_lbl):,}개")
print(f"(기대값 ≈ 유지 쌍 {len(keep_pairs):,}개 + 오류로 보존한 {len(error_pairs):,}개)")

정리 후 남은 이미지: 257,740개
정리 후 남은 라벨:   257,740개
(기대값 ≈ 유지 쌍 89,506개 + 오류로 보존한 0개)
